In [1]:
import cv2
import numpy as np
import dlib
import mediapipe as mp
import albumentations as A

class VideoLipProcessor:
    def __init__(self):
        """Initialize face detectors for extracting lip region."""
        self.detector = dlib.get_frontal_face_detector()
        self.predictor = dlib.shape_predictor("shape_predictor_68_face_landmarks.dat")
        self.mp_face_mesh = mp.solutions.face_mesh
        self.face_mesh = self.mp_face_mesh.FaceMesh(
            static_image_mode=False,
            max_num_faces=1,
            refine_landmarks=True,
            min_detection_confidence=0.5
        )
        
        self.augmentation = A.Compose([
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=10, p=0.5),
            A.RandomBrightnessContrast(p=0.2),
            A.GaussianBlur(blur_limit=(3, 7), p=0.1),
        ])
    
    def extract_lip_region(self, frame):
        """Extracts the lip region using facial landmarks."""
        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = self.detector(gray)

        if len(faces) > 0:
            landmarks = self.predictor(gray, faces[0])
            lip_points = np.array([(landmarks.part(i).x, landmarks.part(i).y) for i in range(48, 68)])
        else:
            results = self.face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if not results.multi_face_landmarks:
                return None
            
            landmarks = results.multi_face_landmarks[0].landmark
            lip_indices = [61, 146, 91, 181, 84, 17, 314, 405, 321, 375, 291]
            lip_points = np.array([(int(landmarks[i].x * frame.shape[1]), int(landmarks[i].y * frame.shape[0])) for i in lip_indices])
        
        x, y, w, h = cv2.boundingRect(lip_points)
        padding = int(w * 0.2)
        x, y = max(0, x - padding), max(0, y - padding)
        w, h = min(frame.shape[1] - x, w + 2*padding), min(frame.shape[0] - y, h + 2*padding)
        
        return frame[y:y+h, x:x+w]

    def preprocess_frame(self, frame, target_size=(100, 50)):
        """Preprocesses a single frame."""
        lip_region = self.extract_lip_region(frame)
        if lip_region is None:
            return None
        
        lip_region = cv2.resize(lip_region, target_size)
        augmented = self.augmentation(image=lip_region)['image']
        return augmented.astype('float32') / 255.0
